# Audio-Lab-PYNQ passthrough debug

Line-in から headphone へ音が出ない場合に、どこで止まっているかを表示します。

In [ ]:
# === Audio-Lab-PYNQ: Line-in -> passthrough -> headphone デバッグ ===
import os
import time
import hashlib
import audio_lab_pynq as aud

EXPECTED_BIT_SHA256 = "54e99977d7ed16d17ac6780d8cdb92199b888dc5f7a85b6214246ec903b97e03"
EXPECTED_HWH_SHA256 = "b9804b61fdf8fa0d74ae27f5635acb427d11e33a71a1476e4046df2890a1f02a"

def banner(title):
    print("\n" + "=" * 72)
    print(title)
    print("=" * 72)

def ok(message):
    print("[OK]", message)

def warn(message):
    print("[WARN]", message)

def fail(message):
    print("[NG]", message)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def b2i(value):
    if isinstance(value, (bytes, bytearray)):
        return int.from_bytes(value, "big")
    if isinstance(value, list):
        out = 0
        for x in value:
            out = (out << 8) | int(x)
        return out
    return int(value)

def reg_hex(value):
    if isinstance(value, (bytes, bytearray)):
        return "0x" + bytes(value).hex()
    if isinstance(value, list):
        return "0x" + bytes(value).hex()
    return hex(int(value))

def dump_codec(codec):
    regs = [
        "R0_CLOCK_CONTROL",
        "R1_PLL_CONTROL",
        "R4_RECORD_MIXER_LEFT_CONTROL_0",
        "R5_RECORD_MIXER_LEFT_CONTROL_1",
        "R6_RECORD_MIXER_RIGHT_CONTROL_0",
        "R7_RECORD_MIXER_RIGHT_CONTROL_1",
        "R15_SERIAL_PORT_CONTROL_0",
        "R19_ADC_CONTROL",
        "R22_PLAYBACK_MIXER_LEFT_CONTROL_0",
        "R24_PLAYBACK_MIXER_RIGHT_CONTROL_0",
        "R29_PLAYBACK_HEADPHONE_LEFT_VOLUME_CONTROL",
        "R30_PLAYBACK_HEADPHONE_RIGHT_VOLUME_CONTROL",
        "R35_PLAYBACK_POWER_MANAGEMENT",
        "R36_DAC_CONTROL_0",
        "R58_SERIAL_INPUT_ROUTE_CONTROL",
        "R59_SERIAL_OUTPUT_ROUTE_CONTROL",
        "R61_DSP_ENABLE",
        "R62_DSP_RUN",
        "R65_CLOCK_ENABLE_0",
        "R66_CLOCK_ENABLE_1",
    ]
    for name in regs:
        try:
            print(f"{name:48s} {reg_hex(getattr(codec, name))}")
        except Exception as e:
            print(f"{name:48s} read failed: {e}")

def dump_switch(name, sw):
    ctrl = sw.read(0x00)
    m0 = sw.read(0x40)
    m1 = sw.read(0x44)
    print(f"{name}.CTRL = {hex(ctrl)}")
    print(f"{name}.M00  = {hex(m0)}")
    print(f"{name}.M01  = {hex(m1)}")
    if ctrl & 0x2:
        ok(f"{name}: configuration committed")
    else:
        ok(f"{name}: route registers readable; CTRL=0 is normal after update")
    return ctrl, m0, m1

def check_switch_expected(src_m0, src_m1, sink_m0, sink_m1):
    if src_m0 == 0 and sink_m0 == 0:
        ok("AXI switch route is line_in -> passthrough -> headphone")
    else:
        fail("AXI switch route is not the expected passthrough/headphone route")
    if src_m1 == 0x80000000 and sink_m1 == 0x80000000:
        ok("Unused AXI switch outputs are disabled")
    else:
        warn("One or more unused AXI switch outputs are not disabled")

banner("1. audio_lab_pynq package and overlay files")
print("audio_lab_pynq module:", aud.__file__)
package_dir = os.path.dirname(aud.__file__)
bit_path = os.path.join(package_dir, "bitstreams", "audio_lab.bit")
hwh_path = os.path.join(package_dir, "bitstreams", "audio_lab.hwh")
print("bit path:", bit_path)
print("hwh path:", hwh_path)

if os.path.exists(bit_path):
    bit_sha = sha256_file(bit_path)
    print("bit sha256:", bit_sha)
    if bit_sha == EXPECTED_BIT_SHA256:
        ok("Expected direct-passthrough bitstream is installed")
    else:
        fail("Different bitstream is installed; PYNQ package may not have been updated")
else:
    fail("bit file not found")

if os.path.exists(hwh_path):
    hwh_sha = sha256_file(hwh_path)
    print("hwh sha256:", hwh_sha)
    if hwh_sha == EXPECTED_HWH_SHA256:
        ok("Expected hwh is installed")
    else:
        fail("Different hwh is installed; metadata may not match bitstream")
else:
    fail("hwh file not found")

banner("2. Overlay load")
ol = aud.AudioLabOverlay()
ok("Overlay loaded")
print("IP dictionary keys:")
for key in sorted(ol.ip_dict.keys()):
    print(" -", key)

if "fx_gain_0" in ol.ip_dict:
    warn("fx_gain_0 still exists in metadata, but the fixed passthrough route should bypass it")
else:
    ok("fx_gain_0 is not present")

banner("3. Codec registers after overlay init")
dump_codec(ol.codec)

banner("4. Re-apply known-good codec path and headphone volume")
# Line-in AUX inputs, ADC/DAC, headphone, serial routing, clocks and DSP run.
ol.codec.R4_RECORD_MIXER_LEFT_CONTROL_0 = 0x01
ol.codec.R5_RECORD_MIXER_LEFT_CONTROL_1 = 0x05
ol.codec.R6_RECORD_MIXER_RIGHT_CONTROL_0 = 0x01
ol.codec.R7_RECORD_MIXER_RIGHT_CONTROL_1 = 0x05
ol.codec.R19_ADC_CONTROL = 0x03
ol.codec.R22_PLAYBACK_MIXER_LEFT_CONTROL_0 = 0x21
ol.codec.R24_PLAYBACK_MIXER_RIGHT_CONTROL_0 = 0x41
ol.codec.R29_PLAYBACK_HEADPHONE_LEFT_VOLUME_CONTROL = 0xE8
ol.codec.R30_PLAYBACK_HEADPHONE_RIGHT_VOLUME_CONTROL = 0xE8
ol.codec.R35_PLAYBACK_POWER_MANAGEMENT = 0x03
ol.codec.R36_DAC_CONTROL_0 = 0x03
ol.codec.R58_SERIAL_INPUT_ROUTE_CONTROL = 0x01
ol.codec.R59_SERIAL_OUTPUT_ROUTE_CONTROL = 0x01
ol.codec.R61_DSP_ENABLE = 0x01
ol.codec.R62_DSP_RUN = 0x01
ol.codec.R65_CLOCK_ENABLE_0 = 0x7F
ol.codec.R66_CLOCK_ENABLE_1 = 0x03
ok("Codec path re-applied")
dump_codec(ol.codec)

banner("5. Route line_in -> passthrough -> headphone")
ol.route(aud.XbarSource.line_in, aud.XbarEffect.passthrough, aud.XbarSink.headphone)
ok("ol.route(line_in, passthrough, headphone) executed")

print("\naxis_switch_source")
_, src_m0, src_m1 = dump_switch("axis_switch_source", ol.axis_switch_source)
print("\naxis_switch_sink")
_, sink_m0, sink_m1 = dump_switch("axis_switch_sink", ol.axis_switch_sink)
check_switch_expected(src_m0, src_m1, sink_m0, sink_m1)

banner("6. Result")
print("If sound is still not continuous:")
print(" - If bit/hwh SHA above is NG, the old overlay is still being loaded.")
print(" - If switch route is NG, Python route control is not taking effect.")
print(" - If both are OK, next check is whether Line-in audio reaches the FPGA by DMA recording.")
print("Now play audio into Line-in and listen on headphone.")


## Optional: DMA input check

ヘッドホンで音が出ない場合、Line-in 信号が FPGA に入っているかを DMA で確認します。

In [ ]:
# === Optional: Line-in が FPGA に届いているか DMA で確認 ===
from pynq import Xlnk
import numpy as np

N = 48000
xlnk = Xlnk()
buf = xlnk.cma_array(shape=(N, 2), dtype=np.int32)
buf[:] = 0

print("Route: line_in -> passthrough -> dma")
ol.route(aud.XbarSource.line_in, aud.XbarEffect.passthrough, aud.XbarSink.dma)

print("Start DMA receive for about 1 second. Play audio into Line-in now.")
ol.axi_dma_0.recvchannel.transfer(buf)
ol.axi_dma_0.recvchannel.wait()

left = buf[:, 0]
right = buf[:, 1]
print("left min/max/peak/nonzero :", int(left.min()), int(left.max()), int(np.abs(left).max()), int(np.count_nonzero(left)))
print("right min/max/peak/nonzero:", int(right.min()), int(right.max()), int(np.abs(right).max()), int(np.count_nonzero(right)))

if np.count_nonzero(left) == 0 and np.count_nonzero(right) == 0:
    print("[NG] DMA capture is all zero. Line-in, codec ADC path, or source route is not receiving signal.")
elif int(np.abs(left).max()) < 100 and int(np.abs(right).max()) < 100:
    print("[WARN] DMA capture is very small. Input volume/source/cable may be too low.")
else:
    print("[OK] DMA capture contains non-zero audio. Input path is alive; problem is likely output/headphone side.")

print("Restore: line_in -> passthrough -> headphone")
ol.route(aud.XbarSource.line_in, aud.XbarEffect.passthrough, aud.XbarSink.headphone)


## Output isolation: DMA tone to headphone

Line-in ではなく DMA から短いテスト音を出します。これで音が聞こえれば headphone/DAC 出力経路は動いています。聞こえなければ出力側の問題です。

In [ ]:
# === Output isolation 1: DMA -> passthrough -> headphone ===
from pynq import Xlnk
import numpy as np
import time

N = 48000
fs = 48000
freq = 440
amplitude = 0x080000

xlnk = Xlnk()
tone = xlnk.cma_array(shape=(N, 2), dtype=np.int32)
wave = (amplitude * np.sin(2 * np.pi * freq * np.arange(N) / fs)).astype(np.int32)
tone[:, 0] = wave
tone[:, 1] = wave

print("========================================================================")
print("Output isolation 1: DMA -> passthrough -> headphone")
print("========================================================================")
print("Tone stats left min/max/peak:", int(tone[:,0].min()), int(tone[:,0].max()), int(np.abs(tone[:,0]).max()))
print("Route: dma -> passthrough -> headphone")
ol.route(aud.XbarSource.dma, aud.XbarEffect.passthrough, aud.XbarSink.headphone)
print("axis_switch_source M00/M01:", hex(ol.axis_switch_source.read(0x40)), hex(ol.axis_switch_source.read(0x44)))
print("axis_switch_sink   M00/M01:", hex(ol.axis_switch_sink.read(0x40)), hex(ol.axis_switch_sink.read(0x44)))

print("Start DMA send. You should hear a 440 Hz tone for about 1 second.")
ol.axi_dma_0.sendchannel.transfer(tone)
ol.axi_dma_0.sendchannel.wait()
print("[DONE] DMA send completed")

print("Restore: line_in -> passthrough -> headphone")
ol.route(aud.XbarSource.line_in, aud.XbarEffect.passthrough, aud.XbarSink.headphone)
print("If the tone was audible but Line-in is not continuous, the output path is OK and the issue is the live passthrough stream/handshake.")
print("If the tone was not audible, the issue is headphone/DAC/output-side routing.")


## Output isolation: Line-in through low-pass to headphone

passthrough ではなく low-pass 経路を headphone に出します。これで連続して聞こえるかを確認します。

In [ ]:
# === Output isolation 2: line_in -> low_pass_filter -> headphone ===
print("========================================================================")
print("Output isolation 2: line_in -> low_pass_filter -> headphone")
print("========================================================================")
ol.route(aud.XbarSource.line_in, aud.XbarEffect.low_pass_filter, aud.XbarSink.headphone)
print("axis_switch_source M00/M01:", hex(ol.axis_switch_source.read(0x40)), hex(ol.axis_switch_source.read(0x44)))
print("axis_switch_sink   M00/M01:", hex(ol.axis_switch_sink.read(0x40)), hex(ol.axis_switch_sink.read(0x44)))
print("Now play audio into Line-in and listen for 5 seconds.")
time.sleep(5)
print("Restore: line_in -> passthrough -> headphone")
ol.route(aud.XbarSource.line_in, aud.XbarEffect.passthrough, aud.XbarSink.headphone)
print("If low-pass is continuous but passthrough is one-shot, the direct passthrough stream path is the failing part.")
print("If low-pass is also one-shot/silent, the failing part is likely i2s_to_stream axis_hp or codec headphone/DAC output.")


## Output-side Codec variation

Codec の output mixer 周辺を少し強めに再設定します。音量に注意してください。

In [ ]:
# === Output-side codec variation ===
print("========================================================================")
print("Output-side codec variation")
print("========================================================================")

# Re-apply DAC/headphone path and slightly louder headphone volume.
ol.codec.R22_PLAYBACK_MIXER_LEFT_CONTROL_0 = 0x21
ol.codec.R24_PLAYBACK_MIXER_RIGHT_CONTROL_0 = 0x41
ol.codec.R29_PLAYBACK_HEADPHONE_LEFT_VOLUME_CONTROL = 0xF0
ol.codec.R30_PLAYBACK_HEADPHONE_RIGHT_VOLUME_CONTROL = 0xF0
ol.codec.R35_PLAYBACK_POWER_MANAGEMENT = 0x03
ol.codec.R36_DAC_CONTROL_0 = 0x03
ol.codec.R58_SERIAL_INPUT_ROUTE_CONTROL = 0x01
ol.codec.R59_SERIAL_OUTPUT_ROUTE_CONTROL = 0x01
ol.codec.R61_DSP_ENABLE = 0x01
ol.codec.R62_DSP_RUN = 0x01
ol.codec.R65_CLOCK_ENABLE_0 = 0x7F
ol.codec.R66_CLOCK_ENABLE_1 = 0x03

print("Codec output registers:")
for name in [
    "R22_PLAYBACK_MIXER_LEFT_CONTROL_0",
    "R24_PLAYBACK_MIXER_RIGHT_CONTROL_0",
    "R29_PLAYBACK_HEADPHONE_LEFT_VOLUME_CONTROL",
    "R30_PLAYBACK_HEADPHONE_RIGHT_VOLUME_CONTROL",
    "R35_PLAYBACK_POWER_MANAGEMENT",
    "R36_DAC_CONTROL_0",
    "R58_SERIAL_INPUT_ROUTE_CONTROL",
    "R59_SERIAL_OUTPUT_ROUTE_CONTROL",
    "R61_DSP_ENABLE",
    "R62_DSP_RUN",
    "R65_CLOCK_ENABLE_0",
    "R66_CLOCK_ENABLE_1",
]:
    print(name, reg_hex(getattr(ol.codec, name)))

ol.route(aud.XbarSource.line_in, aud.XbarEffect.passthrough, aud.XbarSink.headphone)
print("Route restored: line_in -> passthrough -> headphone")
print("Listen again. If this changes nothing, the issue is probably hardware stream handshake/timing, not codec volume.")


## Codec analog bypass tests

FPGA/I2S 出力を使わず、ADAU1761 内部のアナログミキサで Line-in を出力へ回します。これで聞こえる場合、ヘッドホン端子と Codec のアナログ出力は正常で、FPGA から Codec への I2S 出力が原因です。

In [ ]:
# === Codec analog bypass 1: Line-in record mixer -> headphone mixer ===
print("========================================================================")
print("Codec analog bypass 1: record mixer -> headphone mixer")
print("========================================================================")

# Record mixers: LAUX/RAUX to Mixer 1/2 at 0 dB.
ol.codec.R4_RECORD_MIXER_LEFT_CONTROL_0 = 0x01
ol.codec.R5_RECORD_MIXER_LEFT_CONTROL_1 = 0x05
ol.codec.R6_RECORD_MIXER_RIGHT_CONTROL_0 = 0x01
ol.codec.R7_RECORD_MIXER_RIGHT_CONTROL_1 = 0x05

# Playback mixers: enable Mixer 3/4. Keep DAC muted here to isolate analog bypass.
ol.codec.R22_PLAYBACK_MIXER_LEFT_CONTROL_0 = 0x01
ol.codec.R24_PLAYBACK_MIXER_RIGHT_CONTROL_0 = 0x01

# Bypass gains: Mixer1 -> Mixer3 at 0 dB, Mixer2 -> Mixer4 at 0 dB.
# R23 low nibble MX3G1 = 0b0110; R25 high nibble MX4G2 = 0b0110.
ol.codec.R23_PLAYBACK_MIXER_LEFT_CONTROL_1 = 0x06
ol.codec.R25_PLAYBACK_MIXER_RIGHT_CONTROL_1 = 0x60

# Headphone outputs: 0 dB-ish, unmuted, headphone mode enabled.
ol.codec.R29_PLAYBACK_HEADPHONE_LEFT_VOLUME_CONTROL = 0xE7
ol.codec.R30_PLAYBACK_HEADPHONE_RIGHT_VOLUME_CONTROL = 0xE7
ol.codec.R35_PLAYBACK_POWER_MANAGEMENT = 0x03

print("Registers for analog headphone bypass:")
for name in [
    "R4_RECORD_MIXER_LEFT_CONTROL_0",
    "R5_RECORD_MIXER_LEFT_CONTROL_1",
    "R6_RECORD_MIXER_RIGHT_CONTROL_0",
    "R7_RECORD_MIXER_RIGHT_CONTROL_1",
    "R22_PLAYBACK_MIXER_LEFT_CONTROL_0",
    "R23_PLAYBACK_MIXER_LEFT_CONTROL_1",
    "R24_PLAYBACK_MIXER_RIGHT_CONTROL_0",
    "R25_PLAYBACK_MIXER_RIGHT_CONTROL_1",
    "R29_PLAYBACK_HEADPHONE_LEFT_VOLUME_CONTROL",
    "R30_PLAYBACK_HEADPHONE_RIGHT_VOLUME_CONTROL",
    "R35_PLAYBACK_POWER_MANAGEMENT",
]:
    print(name, reg_hex(getattr(ol.codec, name)))

print("Now play audio into Line-in and listen for 5 seconds.")
time.sleep(5)
print("If this is audible, use codec analog bypass or fix FPGA->Codec I2S output.")
print("If this is silent, run the next cell to try the line-output driver path.")


In [ ]:
# === Codec analog bypass 2: try LOUT/ROUT line-output driver path ===
print("========================================================================")
print("Codec analog bypass 2: record mixer -> LOUT/ROUT path")
print("========================================================================")

# Keep the analog bypass into playback mixers enabled.
ol.codec.R4_RECORD_MIXER_LEFT_CONTROL_0 = 0x01
ol.codec.R5_RECORD_MIXER_LEFT_CONTROL_1 = 0x05
ol.codec.R6_RECORD_MIXER_RIGHT_CONTROL_0 = 0x01
ol.codec.R7_RECORD_MIXER_RIGHT_CONTROL_1 = 0x05
ol.codec.R22_PLAYBACK_MIXER_LEFT_CONTROL_0 = 0x01
ol.codec.R23_PLAYBACK_MIXER_LEFT_CONTROL_1 = 0x06
ol.codec.R24_PLAYBACK_MIXER_RIGHT_CONTROL_0 = 0x01
ol.codec.R25_PLAYBACK_MIXER_RIGHT_CONTROL_1 = 0x60

# Mixer 5/6 to line outputs. Some boards wire the jack through LOUT/ROUT rather than LHP/RHP.
ol.codec.R26_PLAYBACK_LR_MIXER_LEFT_LINE_OUTPUT_CONTROL = 0x03
ol.codec.R27_PLAYBACK_LR_MIXER_RIGHT_LINE_OUTPUT_CONTROL = 0x09

# LOUT/ROUT volume: unmute, headphone-capable output mode.
ol.codec.R31_PLAYBACK_LINE_OUTPUT_LEFT_VOLUME_CONTROL = 0xE7
ol.codec.R32_PLAYBACK_LINE_OUTPUT_RIGHT_VOLUME_CONTROL = 0xE7
ol.codec.R35_PLAYBACK_POWER_MANAGEMENT = 0x03

print("Registers for analog line-output bypass:")
for name in [
    "R22_PLAYBACK_MIXER_LEFT_CONTROL_0",
    "R23_PLAYBACK_MIXER_LEFT_CONTROL_1",
    "R24_PLAYBACK_MIXER_RIGHT_CONTROL_0",
    "R25_PLAYBACK_MIXER_RIGHT_CONTROL_1",
    "R26_PLAYBACK_LR_MIXER_LEFT_LINE_OUTPUT_CONTROL",
    "R27_PLAYBACK_LR_MIXER_RIGHT_LINE_OUTPUT_CONTROL",
    "R31_PLAYBACK_LINE_OUTPUT_LEFT_VOLUME_CONTROL",
    "R32_PLAYBACK_LINE_OUTPUT_RIGHT_VOLUME_CONTROL",
    "R35_PLAYBACK_POWER_MANAGEMENT",
]:
    print(name, reg_hex(getattr(ol.codec, name)))

print("Now play audio into Line-in and listen for 5 seconds.")
time.sleep(5)
print("If this is audible, the board jack is likely on LOUT/ROUT; use R31/R32 path.")
print("If both analog bypass tests are silent, the physical output jack/cable/output amplifier path is the next suspect.")


In [ ]:
# === Restore original digital-output codec route ===
print("========================================================================")
print("Restore original digital-output codec route")
print("========================================================================")
ol.codec.R22_PLAYBACK_MIXER_LEFT_CONTROL_0 = 0x21
ol.codec.R23_PLAYBACK_MIXER_LEFT_CONTROL_1 = 0x00
ol.codec.R24_PLAYBACK_MIXER_RIGHT_CONTROL_0 = 0x41
ol.codec.R25_PLAYBACK_MIXER_RIGHT_CONTROL_1 = 0x00
ol.codec.R26_PLAYBACK_LR_MIXER_LEFT_LINE_OUTPUT_CONTROL = 0x00
ol.codec.R27_PLAYBACK_LR_MIXER_RIGHT_LINE_OUTPUT_CONTROL = 0x00
ol.codec.R29_PLAYBACK_HEADPHONE_LEFT_VOLUME_CONTROL = 0xE7
ol.codec.R30_PLAYBACK_HEADPHONE_RIGHT_VOLUME_CONTROL = 0xE7
ol.codec.R31_PLAYBACK_LINE_OUTPUT_LEFT_VOLUME_CONTROL = 0x00
ol.codec.R32_PLAYBACK_LINE_OUTPUT_RIGHT_VOLUME_CONTROL = 0x00
ol.codec.R35_PLAYBACK_POWER_MANAGEMENT = 0x03
ol.codec.R36_DAC_CONTROL_0 = 0x03
ol.route(aud.XbarSource.line_in, aud.XbarEffect.passthrough, aud.XbarSink.headphone)
print("Restored: digital DAC path, line_in -> passthrough -> headphone")
